# ===============================================================
# 🧠 Jenosize Ideas Scraper (Phase 1)
# ===============================================================
# Description:
#   This notebook scrapes English articles from https://www.jenosize.com/en/ideas
#   - Supports per-category scraping (Futurist, Consumer, etc.)
#   - Outputs: data/raw/jenosize_articles.jsonl
# ===============================================================

In [1]:

# ========== 1. Imports ==========
from pathlib import Path
import requests, json, time
from requests.adapters import HTTPAdapter, Retry
import trafilatura
from urllib.parse import urlparse
from dotenv import load_dotenv
import os
from tqdm import tqdm

In [ ]:
# ========== 2. Load environment ==========
from pathlib import Path
import os
from dotenv import load_dotenv

load_dotenv()

PROJECT_ROOT = Path.cwd().resolve()

# โฟลเดอร์หลักสำหรับ raw/processed (อ่านจาก .env ได้, ไม่มีก็ใช้ค่าเริ่มต้นที่รากโปรเจกต์)
RAW_PATH = Path(os.getenv("RAW_PATH", PROJECT_ROOT / "data" / "raw"))
PROCESSED_PATH = Path(os.getenv("PROCESSED_PATH", PROJECT_ROOT / "data" / "processed"))

RAW_PATH.mkdir(parents=True, exist_ok=True)
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

# ไฟล์“เป้าหมายสุดท้าย”
RAW_FILE    = RAW_PATH / "jenosize_articles.jsonl"
RAW_SORTED  = RAW_PATH / "jenosize_articles_sorted.jsonl"
ERROR_LOG   = RAW_PATH / "scrape_errors.log"

In [3]:

# ========== 3. Setup session ==========
def make_session():
    s = requests.Session()
    retries = Retry(total=3, backoff_factor=0.7, status_forcelist=[429, 500, 502, 503, 504])
    s.mount("http://", HTTPAdapter(max_retries=retries))
    s.mount("https://", HTTPAdapter(max_retries=retries))
    s.headers.update({
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/125 Safari/537.36",
        "Accept-Language": "en-US,en;q=0.9,th;q=0.8"
    })
    return s

In [4]:
# ========== 4. Scraper core function ==========
def extract_with_trafilatura(html, url):
    """Extract main article content + metadata"""
    downloaded = trafilatura.extract(html, url=url, include_comments=False,
                                     include_tables=False, favor_precision=True,
                                     with_metadata=True, output_format="json")
    if not downloaded:
        return None
    data = json.loads(downloaded)
    text = (data.get("text") or "").strip()
    title = (data.get("title") or "").strip() or "Untitled"
    if len(text) < 200:
        return None
    return {"url": url, "title": title, "text": text}

def try_fetch(session, url):
    """Handle fetch with fallback (/en/ -> /)"""
    try:
        r = session.get(url, timeout=25)
        if r.status_code == 404:
            return None, "404"
        data = extract_with_trafilatura(r.text, url)
        if data:
            return data, None

        parsed = urlparse(url)
        if "jenosize.com" in parsed.netloc and parsed.path.startswith("/en/"):
            url2 = url.replace("/en/", "/", 1)
            r2 = session.get(url2, timeout=25)
            if r2.status_code == 404:
                return None, "404 (fallback)"
            data2 = extract_with_trafilatura(r2.text, url2)
            if data2:
                return data2, None
            return None, "no text (fallback)"

        return None, "no text"
    except Exception as e:
        return None, str(e)


In [ ]:

# ========== 5. Run scraper per-category ==========
def scrape_category(category_name: str, urls: list[str]):
    session = make_session()
    success, fail = 0, 0

    # ⬇️ ให้เขียน append ลง RAW_FILE (อยู่ที่ data/raw)
    with open(RAW_FILE, "a", encoding="utf-8") as fout:
        for u in tqdm(urls, desc=f"Scraping {category_name}"):
            data, err = try_fetch(session, u)
            if data:
                data["category"] = category_name
                json.dump(data, fout, ensure_ascii=False)
                fout.write("\n")
                success += 1
            else:
                fail += 1
                # ⬇️ ให้ log error ลง ERROR_LOG (อยู่ที่ data/raw)
                with open(ERROR_LOG, "a", encoding="utf-8") as log:
                    log.write(f"{u}\t{err}\n")
            time.sleep(0.7)

    print(f"✅ {category_name}: {success} success, {fail} failed")



In [8]:
# ========== 6. Add category URLs here ==========

FUTURIST = [
   
    "https://www.jenosize.com/en/ideas/futurist/agentic-ai-guide-for-modern-business",
    "https://www.jenosize.com/en/ideas/futurist/ai-customer-service-platform",
    "https://www.jenosize.com/en/ideas/futurist/ai-search-marketing-strategies",
    "https://www.jenosize.com/en/ideas/futurist/behavioral-design-for-marketing",
    "https://www.jenosize.com/en/ideas/futurist/business-model-canvas-2-0",
    "https://www.jenosize.com/en/ideas/futurist/cloud-computing-trends",
    "https://www.jenosize.com/en/ideas/futurist/deepseek-ai-chatbot",
    "https://www.jenosize.com/en/ideas/futurist/digital-twin-business-model",
    "https://www.jenosize.com/en/ideas/futurist/experiential-marketing-2030-trends",
    "https://www.jenosize.com/en/ideas/futurist/hello-jenie-ai",
    "https://www.jenosize.com/en/ideas/futurist/japan-ai-earthquake-disaster-response",
    "https://www.jenosize.com/en/ideas/futurist/laude-4-sonnet-thai-prompts",
    "https://www.jenosize.com/en/ideas/futurist/what-is-quantum-computing",
    "https://www.jenosize.com/en/ideas/futurist/thai-ai-service-platforms",
    "https://www.jenosize.com/en/ideas/futurist/ai-smell-recognition"
    "https://www.jenosize.com/en/ideas/futurist/what-is-critic-gpt",
    "https://www.jenosize.com/en/ideas/futurist/why-your-company-needs-ai-board",
    "https://www.jenosize.com/en/ideas/futurist/what-is-green-business"
]



In [ ]:
CONSUMER = ["https://www.jenosize.com/en/ideas/understand-people-and-consumer/5-dimensions-of-service-quality",
"https://www.jenosize.com/en/ideas/understand-people-and-consumer/brand-awareness-strategy",
"https://www.jenosize.com/en/ideas/understand-people-and-consumer/brand-positioning",
"https://www.jenosize.com/en/ideas/understand-people-and-consumer/buying-cycle-in-retail-business",
"https://www.jenosize.com/en/ideas/understand-people-and-consumer/consumer-buying-roles-marketing-guide",
"https://www.jenosize.com/en/ideas/understand-people-and-consumer/customer-journey-misconceptions",
"https://www.jenosize.com/en/ideas/understand-people-and-consumer/introducing-gen-beta",
"https://www.jenosize.com/en/ideas/understand-people-and-consumer/lead-nurturing-strategy",
"https://www.jenosize.com/en/ideas/understand-people-and-consumer/luxumer-marketing-strategies",
"https://www.jenosize.com/en/ideas/consumer/ai-in-ecommerce"
"https://www.jenosize.com/ideas/understand-people-and-consumer/5-dimensions-of-service-quality",
"https://www.jenosize.com/ideas/understand-people-and-consumer/brand-awareness-strategy",
"https://www.jenosize.com/ideas/understand-people-and-consumer/brand-positioning",
"https://www.jenosize.com/ideas/understand-people-and-consumer/buying-cycle-in-retail-business",
"https://www.jenosize.com/ideas/understand-people-and-consumer/consumer-buying-roles-marketing-guide",
"https://www.jenosize.com/ideas/understand-people-and-consumer/customer-journey-misconceptions",
"https://www.jenosize.com/ideas/understand-people-and-consumer/introducing-gen-beta",
"https://www.jenosize.com/ideas/understand-people-and-consumer/lead-nurturing-strategy",
"https://www.jenosize.com/ideas/understand-people-and-consumer/luxumer-marketing-strategies",

"https://www.jenosize.com/ideas/understand-people-and-consumer/reciprocity-marketing",
"https://www.jenosize.com/ideas/understand-people-and-consumer/single-customer-view",
"https://www.jenosize.com/ideas/understand-people-and-consumer/stp-marketing-strategy-guide"]

In [ ]:
TRANSFORMATION = [ "https://www.jenosize.com/en/ideas/transformation-and-technology/big-data-in-government",
"https://www.jenosize.com/en/ideas/transformation-and-technology/digital-inclusion",
"https://www.jenosize.com/en/ideas/transformation-and-technology/digital-transformation-in-government",
"https://www.jenosize.com/en/ideas/transformation-and-technology/immersive-event-trend",
"https://www.jenosize.com/en/ideas/transformation-and-technology/low-code-vs-no-code-platform",
"https://www.jenosize.com/en/ideas/transformation-and-technology/loyalty-program-tips",
"https://www.jenosize.com/en/ideas/transformation-and-technology/marketplace-vs-brandcom-vs-social-commerce",
"https://www.jenosize.com/en/ideas/transformation-and-technology/online-booking-system-for-service-business",
"https://www.jenosize.com/en/ideas/transformation-and-technology/smart-city-transformation",
"https://www.jenosize.com/en/ideas/transformation-and-technology/super-app-vs-regular-app",
"https://www.jenosize.com/en/ideas/transformation-and-technology/technostalgia-marketing",
"https://www.jenosize.com/en/ideas/transformation-and-technology/ux-research-for-businesses",
"https://www.jenosize.com/en/ideas/transformation-and-technology/big-data-in-government",
"https://www.jenosize.com/en/ideas/transformation-and-technology/digital-inclusion",
"https://www.jenosize.com/en/ideas/transformation-and-technology/digital-transformation-in-government",
"https://www.jenosize.com/en/ideas/transformation-and-technology/immersive-event-trend",
"https://www.jenosize.com/en/ideas/transformation-and-technology/low-code-vs-no-code-platform",
"https://www.jenosize.com/en/ideas/transformation-and-technology/loyalty-program-tips",
"https://www.jenosize.com/en/ideas/transformation-and-technology/marketplace-vs-brandcom-vs-social-commerce",
"https://www.jenosize.com/en/ideas/transformation-and-technology/online-booking-system-for-service-business",
"https://www.jenosize.com/en/ideas/transformation-and-technology/smart-city-transformation",
"https://www.jenosize.com/en/ideas/transformation-and-technology/super-app-vs-regular-app",
"https://www.jenosize.com/en/ideas/transformation-and-technology/technostalgia-marketing",
"https://www.jenosize.com/en/ideas/transformation-and-technology/ux-research-for-businesses" ]

In [14]:
UTILITIES = [
    "https://www.jenosize.com/ideas/utility-for-our-world/5-green-energies-for-businesses",
    "https://www.jenosize.com/ideas/utility-for-our-world/carbon-credits-business-opportunities",
    "https://www.jenosize.com/ideas/utility-for-our-world/esg-green-technology",
    "https://www.jenosize.com/ideas/utility-for-our-world/fast-fashion-trend",
    "https://www.jenosize.com/ideas/utility-for-our-world/global-risks-business-preparedness",
    "https://www.jenosize.com/ideas/utility-for-our-world/green-intelligence-and-sustainability",
    "https://www.jenosize.com/ideas/utility-for-our-world/green-silver-blue-economy",
    "https://www.jenosize.com/ideas/utility-for-our-world/how-online-booking-helps-reduce-co2",
    "https://www.jenosize.com/ideas/utility-for-our-world/improving-sustainability-in-business",
    "https://www.jenosize.com/ideas/utility-for-our-world/plant-based-food-business",
    "https://www.jenosize.com/ideas/utility-for-our-world/regenerative-business",           
    "https://www.jenosize.com/ideas/utility-for-our-world/what-is-e-waste" ]


In [17]:

MARKETING = [ "https://www.jenosize.com/en/ideas/real-time-marketing/ai-algorithm-friendly-content",
 "https://www.jenosize.com/en/ideas/real-time-marketing/brand-loyalty-pyramid",
 "https://www.jenosize.com/en/ideas/real-time-marketing/data-driven-campaign-tips",
 "https://www.jenosize.com/en/ideas/real-time-marketing/event-booth-tips",
 "https://www.jenosize.com/en/ideas/real-time-marketing/event-marketing-strategy",
 "https://www.jenosize.com/en/ideas/real-time-marketing/gamification-o2o-strategy",
 "https://www.jenosize.com/en/ideas/real-time-marketing/live-commerce-techniques-to-boost-sales",
 "https://www.jenosize.com/en/ideas/real-time-marketing/micro-moment-marketing",
 "https://www.jenosize.com/en/ideas/real-time-marketing/mrbeast-video-marketing-tips",
 "https://www.jenosize.com/en/ideas/real-time-marketing/rebranding-strategy",
 "https://www.jenosize.com/en/ideas/real-time-marketing/seo-vs-sem-vs-smm",
 "https://www.jenosize.com/en/ideas/real-time-marketing/storytelling-techniques",
 "https://www.jenosize.com/ideas/real-time-marketing/ai-algorithm-friendly-content",
 "https://www.jenosize.com/ideas/real-time-marketing/brand-loyalty-pyramid",
 "https://www.jenosize.com/ideas/real-time-marketing/data-driven-campaign-tips",
 "https://www.jenosize.com/ideas/real-time-marketing/event-booth-tips",
 "https://www.jenosize.com/ideas/real-time-marketing/event-marketing-strategy",
 "https://www.jenosize.com/ideas/real-time-marketing/gamification-o2o-strategy",
 "https://www.jenosize.com/ideas/real-time-marketing/live-commerce-techniques-to-boost-sales",
 "https://www.jenosize.com/ideas/real-time-marketing/micro-moment-marketing",
 "https://www.jenosize.com/ideas/real-time-marketing/mrbeast-video-marketing-tips",
 "https://www.jenosize.com/ideas/real-time-marketing/rebranding-strategy",
 "https://www.jenosize.com/ideas/real-time-marketing/seo-vs-sem-vs-smm",
 "https://www.jenosize.com/ideas/real-time-marketing/storytelling-techniques"]

In [18]:
EXPERIENCE = [ "https://www.jenosize.com/en/ideas/experience-the-new-world/5g-impact-on-business",
              "https://www.jenosize.com/en/ideas/experience-the-new-world/event-design-thinking",
              "https://www.jenosize.com/en/ideas/experience-the-new-world/okr-vs-kpi-difference",
              "https://www.jenosize.com/en/ideas/experience-the-new-world/sensory-experience-event-strategy",
              "https://www.jenosize.com/en/ideas/experience-the-new-world/build-your-brand-without-marketplace",
              "https://www.jenosize.com/en/ideas/experience-the-new-world/time-management-techniques",
              "https://www.jenosize.com/en/ideas/experience-the-new-world/top-startup-trends",
              "https://www.jenosize.com/en/ideas/experience-the-new-world/oligopoly-market-in-thailand",
              "https://www.jenosize.com/en/ideas/experience-the-new-world/is-hybrid-working-the-new-norm",
              "https://www.jenosize.com/en/ideas/experience-the-new-world/what-is-lipstick-effect",
              "https://www.jenosize.com/en/ideas/experience-the-new-world/7-megatrends-in-2024",
              "https://www.jenosize.com/en/ideas/experience-the-new-world/customer-experience-for-o2o-marketing",
              "https://www.jenosize.com/en/ideas/experience-the-new-world/spatial-computing-and-immersive-tech"
]

In [9]:


# ========== 7. Run scraping per category ==========
# (รันทีละหมวดเพื่อป้องกันเว็บล่ม)
scrape_category("Futurist", FUTURIST)
# ต่อด้วยหมวดอื่นเมื่อพร้อม


Scraping Futurist: 100%|██████████| 17/17 [01:36<00:00,  5.65s/it]

✅ Futurist: 16 success, 1 failed


In [11]:

scrape_category("Understand People & Consumer", CONSUMER)

Scraping Understand People & Consumer: 100%|██████████| 24/24 [01:58<00:00,  4.95s/it]

✅ Understand People & Consumer: 20 success, 4 failed


In [23]:
scrape_category("Transformation & Technology", TRANSFORMATION)

Scraping Transformation & Technology: 100%|██████████| 27/27 [04:49<00:00, 10.71s/it]

✅ Transformation & Technology: 22 success, 5 failed


In [22]:
scrape_category("Utility for Our World", UTILITIES)

Scraping Utility for Our World: 100%|██████████| 12/12 [01:42<00:00,  8.57s/it]

✅ Utility for Our World: 11 success, 1 failed


In [20]:

scrape_category("Real-time Marketing", MARKETING)

Scraping Real-time Marketing: 100%|██████████| 24/24 [05:49<00:00, 14.55s/it]

✅ Real-time Marketing: 20 success, 4 failed


In [21]:

scrape_category("Experience the New World", EXPERIENCE)

Scraping Experience the New World: 100%|██████████| 13/13 [02:11<00:00, 10.11s/it]

✅ Experience the New World: 12 success, 1 failed


In [24]:

import json
from pathlib import Path

in_file = Path("data/raw/jenosize_articles.jsonl")
out_file = Path("data/raw/jenosize_articles_sorted.jsonl")

with open(in_file, "r", encoding="utf-8") as f:
    data = [json.loads(line) for line in f if line.strip()]



In [25]:
# Cell 2
data_sorted = sorted(data, key=lambda x: (x.get("category", ""), x.get("title", "")))

with open(out_file, "w", encoding="utf-8") as f:
    for item in data_sorted:
        json.dump(item, f, ensure_ascii=False)
        f.write("\n")

print(f"✅ Sorted & saved: {out_file.resolve()}")


✅ Sorted & saved: D:\mini-jane-demo\notebooks\data\raw\jenosize_articles_sorted.jsonl
